In [ ]:
# Import necessary libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense
import matplotlib.pyplot as plt

# Generate synthetic time series data
def generate_time_series(n_steps):
    time = np.arange(0, n_steps)
    data = np.sin(0.1 * time) + np.random.randn(n_steps) * 0.1  # Sine wave with noise
    return data

# Prepare the dataset
n_steps = 1000
time_series = generate_time_series(n_steps)

# Split the data into input (X) and target (y)
def split_sequence(sequence, n_steps_in, n_steps_out):
    X, y = [], []
    for i in range(len(sequence)):
        end_ix = i + n_steps_in
        out_end_ix = end_ix + n_steps_out
        if out_end_ix > len(sequence):
            break
        X.append(sequence[i:end_ix])
        y.append(sequence[end_ix:out_end_ix])
    return np.array(X), np.array(y)

n_steps_in = 50  # Number of input time steps
n_steps_out = 10  # Number of output time steps
X, y = split_sequence(time_series, n_steps_in, n_steps_out)

# Reshape X to be compatible with RNN input format: [samples, timesteps, features]
X = X.reshape((X.shape[0], X.shape[1], 1))

# Split into training and testing sets
train_size = int(0.8 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Build the RNN model
model = Sequential([
    SimpleRNN(50, activation='relu', input_shape=(n_steps_in, 1)),  # RNN layer with 50 units
    Dense(n_steps_out)  # Output layer
])

# Compile the model
model.compile(optimizer='adam', loss='mse')

# Train the model
history = model.fit(X_train, y_train, epochs=20, validation_data=(X_test, y_test))

# Evaluate the model
loss = model.evaluate(X_test, y_test)
print(f"Test Loss (MSE): {loss:.4f}")

# Make predictions
y_pred = model.predict(X_test)

# Plot the results
plt.figure(figsize=(12, 6))
plt.plot(y_test[0], label='Actual')
plt.plot(y_pred[0], label='Predicted')
plt.title("RNN Time Series Prediction")
plt.xlabel("Time Steps")
plt.ylabel("Value")
plt.legend()
plt.show()
